In [8]:
	
import pandas as pd 
import re
from rapidfuzz import fuzz

In [9]:
df = pd.read_csv("/home/admin-groep11/CVMatching-1/data/raw/Resume.csv")

df_klein = df.sample(frac=0.4, random_state=42)

df_klein.to_csv("/home/admin-groep11/CVMatching-1/data/raw/Resume_short.csv", index=False)

In [ ]:
df1 = pd.read_csv('/home/admin-groep11/CVMatching-1/data/raw/job_descriptions/job_descriptions.csv')

In [10]:
df2 = pd.read_csv('/home/admin-groep11/CVMatching-1/data/raw/job_descriptions/job_descriptions2.csv')

In [11]:
df3 = pd.read_csv('/home/admin-groep11/CVMatching-1/data/raw/Resume_short.csv')

In [13]:
jobs_df = df2[['Job ID', 'Business Title', 'Job Description', 'Job Category']].fillna('')
resumes_df = df3[['ID', 'Resume_str', 'Category']].fillna('')
data_list = []


for job_index, job_row in jobs_df.iterrows():
    for resume_index, resume_row in resumes_df.iterrows():
        is_match = 0
        job_category_and_title = (job_row['Job Category'] + ' ' + job_row['Business Title']).lower()
        resume_category = resume_row['Category'].lower()
        if resume_category in job_category_and_title:
            is_match = 1

        data_list.append({
            'job_id': job_row['Job ID'],
            'resume_id': resume_row['ID'],
            'job_text': job_row['Job Description'],
            'resume_text': resume_row['Resume_str'],
            'label': is_match
        })
labeled_df = pd.DataFrame(data_list)

In [7]:
print("Eerste 5 rijen van de gelabelde dataset:")
print(labeled_df.head())
print("\nVerdeling van de labels:")
print(labeled_df['label'].value_counts())
labeled_df.to_csv('gelabelde_data.csv', index=False)

Eerste 5 rijen van de gelabelde dataset:
   job_id  resume_id                                           job_text  \
0   87990   99244405  Division of Economic & Financial Opportunity (...   
1   87990   17562754  Division of Economic & Financial Opportunity (...   
2   87990   30311725  Division of Economic & Financial Opportunity (...   
3   87990   19007667  Division of Economic & Financial Opportunity (...   
4   87990   11065180  Division of Economic & Financial Opportunity (...   

                                         resume_text  label  
0             Kpandipou    Koffi         Summary ...      0  
1           DIRECTOR OF DIGITAL TRANSFORMATION   ...      0  
2           SENIOR PROJECT MANAGER       Professi...      0  
3           CHEF       Summary     Experienced ca...      0  
4           OPERATIONS MANAGER       Summary    E...      0  

Verdeling van de labels:
label
0    4192287
1      55353
Name: count, dtype: int64


OSError: [Errno 28] No space left on device

In [ ]:
df4 = pd.read_csv("/home/admin-groep11/CVMatching-1/data/raw/Resume_short.csv")

In [ ]:
# eerste dataset
jobs_df = df2[['Job ID', 'Business Title', 'Job Description', 'Job Category']].fillna('')
resumes_df = df4[['ID', 'Resume_str', 'Category']].fillna('')
data_list = []


for job_index, job_row in jobs_df.iterrows():
    for resume_index, resume_row in resumes_df.iterrows():
        is_match = 0
        job_category_and_title = (job_row['Job Category'] + ' ' + job_row['Business Title']).lower()
        resume_category = resume_row['Category'].lower()
        if resume_category in job_category_and_title:
            is_match = 1

        data_list.append({
            'job_id': job_row['Job ID'],
            'resume_id': resume_row['ID'],
            'job_text': job_row['Job Description'],
            'resume_text': resume_row['Resume_str'],
            'label': is_match
        })
labeled_df = pd.DataFrame(data_list)

In [ ]:
labeled_df1 = pd.DataFrame(data_list)

# opslaan
labeled_df1.to_csv("/home/admin-groep11/CVMatching-1/data/processed/labeled_jobdescription1", index=False, encoding="utf-8")

print("Dataset opgeslagen als labeled_jobs1_resumes.csv")

In [ ]:
jobs_df = df2[['Job ID', 'Business Title', 'Job Description', 'Job Category']].fillna('')
resumes_df = df4[['ID', 'Resume_str', 'Category']].fillna('')

data_list = []

# Functie om regex + fuzzy matching te combineren
def is_category_match(resume_category, job_category_and_title):
    # Regex: spaties, underscores en koppeltekens als varianten toestaan
    pattern = resume_category.replace(" ", "[-_ ]?")
    if re.search(pattern, job_category_and_title, re.IGNORECASE):
        return True

    # Fuzzy: tolerant voor spelfouten
    score = fuzz.partial_ratio(resume_category.lower(), job_category_and_title.lower())
    if score > 60:  # threshold kun je tunen
        return True

    return False

# Loop door de dataframes heen
for job_index, job_row in jobs_df.iterrows():
    for resume_index, resume_row in resumes_df.iterrows():
        is_match = 0
        job_category_and_title = (job_row['Job Category'] + ' ' + job_row['Business Title']).lower()
        resume_category = resume_row['Category'].lower()

        if is_category_match(resume_category, job_category_and_title):
            is_match = 1

        data_list.append({
            'job_id': job_row['Job ID'],
            'resume_id': resume_row['ID'],
            'job_text': job_row['Job Description'],
            'resume_text': resume_row['Resume_str'],
            'label': is_match
        })

# Maak de gelabelde dataset
labeled_df = pd.DataFrame(data_list)

In [14]:
labeled_df.to_csv("/home/admin-groep11/CVMatching-1/data/processed/labeled_jobdescriptions2.csv", index=False, encoding="utf-8")

print("geslaagd")

geslaagd


In [15]:
# Tel hoeveel matches (1) en non-matches (0) er zijn
label_counts = labeled_df['label'].value_counts()

print("Aantal labels per categorie:")
print(label_counts)

Aantal labels per categorie:
label
0    3357502
1      41978
Name: count, dtype: int64


In [17]:
# === Stap 1: Balanceren ===
min_class_size = labeled_df['label'].value_counts().min()

balanced_df = pd.concat([
    labeled_df[labeled_df['label'] == 0].sample(n=min_class_size, random_state=42),
    labeled_df[labeled_df['label'] == 1].sample(n=min_class_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Dataset gebalanceerd: {len(balanced_df)} rijen totaal ({min_class_size} per klasse)")
print(balanced_df['label'].value_counts())

# === Stap 2: Opslaan ===
balanced_df.to_csv(
    "/home/admin-groep11/CVMatching-1/data/processed/labeled_jobdescriptions2.csv",
    index=False,
    encoding="utf-8"
)

print("💾 Gebalanceerde dataset opgeslagen.")



✅ Dataset gebalanceerd: 83956 rijen totaal (41978 per klasse)
label
0    41978
1    41978
Name: count, dtype: int64
💾 Gebalanceerde dataset opgeslagen.
